# CS F425 Deep Learning: Assignment II
## Step 1: Source Data Unification, SAM Geometry Distillation & DSLR-Style Augmentation
**Course Group**: G09 | **Target**: 60/60 Marks

This notebook runs **Step 1** of the Master Execution Plan on Kaggle GPU:
1. **Graph-W (37 train boards)**: Box prompt distillation with frozen SAM (`vit_h`) + 25-class inheritance.
2. **Graph-F (115 train boards)**: Touching passive blob splitting + distance transform watershed / SAM.
3. **Rule T3 Quality Control**: Same-class Mask-NMS, ascending area sorting (small passives on top), pure black background `0`.
4. **Deterministic Split**: 152 boards partitioned by physical PCB ID into **131 train** / **21 val** boards.
5. **1024x1024 Tiling**: 256px overlap (stride 768) + DSLR Photometric Domain Randomization.

### 1. Environment & GPU Verification

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM Total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

### 2. Install Dependencies & Download SAM vit_h Weights

In [ ]:
!pip install git+https://github.com/facebookresearch/segment-anything.git
!pip install opencv-python-headless pillow scipy matplotlib tqdm

import os
os.makedirs('checkpoints', exist_ok=True)
sam_ckpt = 'checkpoints/sam_vit_h_4b8939.pth'
if not os.path.isfile(sam_ckpt) or os.path.getsize(sam_ckpt) < 1_000_000_000:
    print('Downloading SAM vit_h weights (2.56 GB)...')
    !wget -c --progress=bar:force:noscroll https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O {sam_ckpt}
print('SAM vit_h Checkpoint ready!')

### 3. Check Dataset Attachment
Ensure `irislan/graphpcb` is attached in the Kaggle notebook (sidebar -> Add Input -> Search 'graphpcb').

In [ ]:
import os, glob

# Auto-discover GraphPCB dataset across /kaggle/input
source_dir = None
if os.path.isdir('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'Graph-W' in dirs and 'Graph-F' in dirs:
            source_dir = root
            break

if source_dir is None:
    possible_paths = [
        '/kaggle/input/the-graphpcb-dataset/GraphPCB',
        '/kaggle/input/the-graphpcb-dataset',
        '/kaggle/input/graphpcb/GraphPCB',
        '/kaggle/input/graphpcb',
        'data/GraphPCB',
        '../data/GraphPCB'
    ]
    for p in possible_paths:
        if os.path.isdir(os.path.join(p, 'Graph-W')) and os.path.isdir(os.path.join(p, 'Graph-F')):
            source_dir = p
            break

assert source_dir is not None, f'Could not find GraphPCB dataset. Contents of /kaggle/input: {os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "N/A"}'
print(f'Located GraphPCB dataset at: {source_dir}')

### 4. Run Step 1.1: Teacher SAM Distillation
Converts 37 Graph-W boards and 115 Graph-F boards into crisp, geometry-aware instance masks with Rule T3 Quality Control.

In [ ]:
!python3 -u scripts/offline_teacher_source.py \
    --source "{source_dir}" \
    --sam-ckpt "{sam_ckpt}" \
    --sam-type vit_h \
    --out "data/distilled_source" \
    --device auto \
    --batch-size 32 \
    --mask-nms-thresh 0.50

### 5. Run Step 1.2: Deterministic 131/21 Board Split & 1024x1024 Tiling
Splits boards strictly by physical board identity to prevent cross-view leakage, and extracts 1024x1024 overlapping tiles.

In [ ]:
!python3 -u src/semantic/prepare_source.py \
    --source "data/distilled_source" \
    --out "data/prepared_tiles" \
    --seed 42 \
    --tile 1024 \
    --overlap 256 \
    --target-val-boards 21

### 6. Verify Outputs & Visual Inspection

In [ ]:
import json, glob
import matplotlib.pyplot as plt
import cv2

manifest_path = 'data/prepared_tiles/manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)

print('Total boards:', manifest['total_source_boards'])
print('Train boards:', manifest['train_boards_count'], '| Val boards:', manifest['val_boards_count'])
print('Train tiles:', manifest['train_tiles_count'], '| Val tiles:', manifest['val_tiles_count'])

# Display sample tile and its corresponding ground-truth mask
sample_tiles = glob.glob('data/prepared_tiles/images/train/*.jpg')
if sample_tiles:
    sample_img_p = sample_tiles[0]
    sample_mask_p = sample_img_p.replace('images', 'masks').replace('.jpg', '.png')
    
    img = cv2.cvtColor(cv2.imread(sample_img_p), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(sample_mask_p, cv2.IMREAD_UNCHANGED)
    
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f'Sample RGB Tile (1024x1024): {os.path.basename(sample_img_p)}')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='tab20')
    plt.title('Corresponding Distilled Mask')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

### 7. Compress Step 1 Output for Next Steps

In [ ]:
!tar -czf prepared_tiles_step1.tar.gz data/prepared_tiles
print('Exported prepared_tiles_step1.tar.gz successfully!')
!ls -lh prepared_tiles_step1.tar.gz